# YOLO License Plate Detector Training

Notebook ini melatih YOLO untuk menemukan **lokasi plat nomor**. Model ini tidak membaca karakter; crop hasil deteksi diteruskan ke OCR/recognizer dari notebook `01_license_plate_detection_and_ocr_baseline.ipynb`.

Pipeline: **Pascal VOC XML -> audit -> fixed split -> YOLO dataset -> train -> validate -> predict -> export**.

> Dataset saat ini banyak berisi crop plat yang memenuhi hampir seluruh frame. Model baseline tetap dapat dilatih, tetapi generalisasi ke IP Camera harus diuji dan ditingkatkan menggunakan foto kendaraan utuh dari sudut kamera aktual.

## 0. Environment
Pilih kernel `Python (.venv) - Smart Parking System`. Training GPU memerlukan PyTorch CUDA; environment saat ini otomatis menggunakan CPU bila CUDA tidak tersedia.

In [ ]:
import csv
import hashlib
import random
import shutil
import statistics
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import torch
import yaml
from IPython.display import Image as DisplayImage, display
from ultralytics import YOLO

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Configuration and paths

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'apps' / 'vision').is_dir() and (candidate / 'ml').is_dir():
            return candidate
    raise FileNotFoundError('Smart Parking System project root was not found.')


PROJECT_ROOT = find_project_root(Path.cwd())
SOURCE_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'indonesian-plate-number' / 'plate_number_recognition_datasets_v1'
SOURCE_IMAGE_ROOT = SOURCE_ROOT / 'images'
SOURCE_ANNOTATION_ROOT = SOURCE_ROOT / 'annotations'
YOLO_DATASET_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'yolo_license_plate'
DATASET_YAML = YOLO_DATASET_ROOT / 'dataset.yaml'
MANIFEST_CSV = YOLO_DATASET_ROOT / 'manifest.csv'
BASE_MODEL_PATH = PROJECT_ROOT / 'ml' / 'weights' / 'yolo11n.pt'
EXPERIMENT_ROOT = PROJECT_ROOT / 'ml' / 'experiments' / 'yolo_license_plate_detector'
RUN_NAME = 'yolo11n_plate_v1'

SEED = 42
VALIDATION_RATIO = 0.20
IMAGE_SIZE = 640
EPOCHS = 50
BATCH_SIZE = 8
PATIENCE = 15
TRAINING_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
RUN_TRAINING = False  # best.pt sudah tersedia; ubah True hanya untuk retraining.
RUN_EXPORT = False

if not SOURCE_IMAGE_ROOT.is_dir() or not SOURCE_ANNOTATION_ROOT.is_dir():
    raise FileNotFoundError(f'Source dataset is incomplete: {SOURCE_ROOT}')
if not BASE_MODEL_PATH.is_file():
    raise FileNotFoundError(f'Pretrained model is missing: {BASE_MODEL_PATH}')

print(f'Source: {SOURCE_ROOT}')
print(f'Prepared dataset: {YOLO_DATASET_ROOT}')
print(f'Device: {TRAINING_DEVICE}')
print(f'RUN_TRAINING: {RUN_TRAINING}')

## 2. Parse and audit Pascal VOC annotations
Bounding box divalidasi terhadap ukuran gambar. `scene` berarti plat tidak mendominasi frame; `plate_crop` berarti setidaknya satu box memakai lebih dari 50% area frame.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def parse_voc_annotation(xml_path: Path) -> dict:
    root = ET.parse(xml_path).getroot()
    width = int(float(root.findtext('size/width', '0')))
    height = int(float(root.findtext('size/height', '0')))
    boxes = []
    for obj in root.findall('object'):
        if (obj.findtext('name') or '').strip().lower() != 'plate':
            continue
        bbox = obj.find('bndbox')
        if bbox is None:
            continue
        x1, y1, x2, y2 = [float(bbox.findtext(key, '0')) for key in ('xmin', 'ymin', 'xmax', 'ymax')]
        x1, y1 = max(0.0, x1), max(0.0, y1)
        x2, y2 = min(float(width), x2), min(float(height), y2)
        if width > 0 and height > 0 and x2 > x1 and y2 > y1:
            boxes.append((x1, y1, x2, y2))
    return {'width': width, 'height': height, 'boxes': boxes}


def source_records(split: str) -> list[dict]:
    records = []
    for image_path in sorted((SOURCE_IMAGE_ROOT / split).iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        xml_path = SOURCE_ANNOTATION_ROOT / split / f'{image_path.stem}.xml'
        if not xml_path.is_file():
            raise FileNotFoundError(f'Missing annotation: {xml_path}')
        annotation = parse_voc_annotation(xml_path)
        if not annotation['boxes']:
            raise ValueError(f'No valid plate boxes: {xml_path}')
        area_ratios = [((x2 - x1) * (y2 - y1)) / (annotation['width'] * annotation['height']) for x1, y1, x2, y2 in annotation['boxes']]
        records.append({
            'source_split': split, 'image_path': image_path, 'xml_path': xml_path,
            'width': annotation['width'], 'height': annotation['height'], 'boxes': annotation['boxes'],
            'max_area_ratio': max(area_ratios), 'scene_type': 'plate_crop' if max(area_ratios) > 0.50 else 'scene',
        })
    return records


train_source_records = source_records('train')
test_source_records = source_records('test')
all_source_records = train_source_records + test_source_records
all_ratios = [record['max_area_ratio'] for record in all_source_records]
print(f'Images: {len(all_source_records):,}')
print(f'Boxes: {sum(len(record["boxes"]) for record in all_source_records):,}')
print(f'Scene types: {dict(Counter(record["scene_type"] for record in all_source_records))}')
print(f'Median maximum box area: {statistics.median(all_ratios):.2%}')

## 3. Create deterministic stratified split
Folder Kaggle `test` dipertahankan sebagai final test. Folder `train` dibagi menjadi train/validation secara deterministik dan distratifikasi berdasarkan `scene_type`.

In [ ]:
def stratified_train_val_split(records: list[dict], validation_ratio: float, seed: int) -> tuple[list[dict], list[dict]]:
    rng = random.Random(seed)
    grouped = {}
    for record in records:
        grouped.setdefault(record['scene_type'], []).append(record)
    train_records, val_records = [], []
    for group in grouped.values():
        group = group.copy()
        rng.shuffle(group)
        val_count = max(1, round(len(group) * validation_ratio))
        val_records.extend(group[:val_count])
        train_records.extend(group[val_count:])
    rng.shuffle(train_records)
    rng.shuffle(val_records)
    return train_records, val_records


train_records, val_records = stratified_train_val_split(train_source_records, VALIDATION_RATIO, SEED)
split_records = {'train': train_records, 'val': val_records, 'test': test_source_records}
for split, records_for_split in split_records.items():
    distribution = Counter(record['scene_type'] for record in records_for_split)
    print(f'{split:5s}: {len(records_for_split):3d} images | {dict(distribution)}')

content_hash_splits = {}
for split, records_for_split in split_records.items():
    for record in records_for_split:
        digest = hashlib.sha256(record['image_path'].read_bytes()).hexdigest()
        content_hash_splits.setdefault(digest, set()).add(split)
cross_split_duplicates = [splits for splits in content_hash_splits.values() if len(splits) > 1]
if cross_split_duplicates:
    raise RuntimeError(f'Found {len(cross_split_duplicates)} duplicate image group(s) across splits.')
print('Cross-split duplicate images: 0')

## 4. Convert dataset to YOLO format
Dataset hasil konversi berada di `data/processed/yolo_license_plate`. Manifest mempertahankan hubungan ke file sumber. Menjalankan ulang cell ini akan mengganti hanya folder prepared dataset tersebut.

In [ ]:
def voc_box_to_yolo(box: tuple[float, float, float, float], width: int, height: int) -> tuple[float, float, float, float]:
    x1, y1, x2, y2 = box
    return ((x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height)


def safe_sample_name(record: dict, split: str, index: int) -> str:
    digest = hashlib.sha1(str(record['image_path']).encode('utf-8')).hexdigest()[:8]
    return f'{split}_{index:04d}_{digest}'


if YOLO_DATASET_ROOT.exists():
    shutil.rmtree(YOLO_DATASET_ROOT)

manifest_rows = []
for split, records_for_split in split_records.items():
    image_output_dir = YOLO_DATASET_ROOT / 'images' / split
    label_output_dir = YOLO_DATASET_ROOT / 'labels' / split
    image_output_dir.mkdir(parents=True, exist_ok=True)
    label_output_dir.mkdir(parents=True, exist_ok=True)
    for index, record in enumerate(records_for_split, start=1):
        sample_name = safe_sample_name(record, split, index)
        target_image = image_output_dir / f'{sample_name}{record["image_path"].suffix.lower()}'
        target_label = label_output_dir / f'{sample_name}.txt'
        shutil.copy2(record['image_path'], target_image)
        label_lines = []
        for box in record['boxes']:
            x_center, y_center, box_width, box_height = voc_box_to_yolo(box, record['width'], record['height'])
            label_lines.append(f'0 {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}')
        target_label.write_text('\n'.join(label_lines) + '\n', encoding='ascii')
        manifest_rows.append({
            'split': split, 'prepared_image': str(target_image.relative_to(YOLO_DATASET_ROOT)),
            'source_image': str(record['image_path'].relative_to(PROJECT_ROOT)),
            'scene_type': record['scene_type'], 'box_count': len(record['boxes']),
        })

dataset_config = {
    'path': str(YOLO_DATASET_ROOT.resolve()), 'train': 'images/train', 'val': 'images/val',
    'test': 'images/test', 'names': {0: 'license_plate'},
}
DATASET_YAML.write_text(yaml.safe_dump(dataset_config, sort_keys=False), encoding='utf-8')
with MANIFEST_CSV.open('w', newline='', encoding='utf-8-sig') as handle:
    writer = csv.DictWriter(handle, fieldnames=manifest_rows[0].keys())
    writer.writeheader()
    writer.writerows(manifest_rows)
print(f'Prepared {len(manifest_rows):,} images')
print(f'Dataset YAML: {DATASET_YAML}')
print(f'Manifest: {MANIFEST_CSV}')

## 5. Validate and visualize converted labels

In [ ]:
def draw_yolo_labels(image_path: Path, label_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f'Unreadable image: {image_path}')
    height, width = image.shape[:2]
    for line in label_path.read_text(encoding='ascii').splitlines():
        class_id, x_center, y_center, box_width, box_height = map(float, line.split())
        x1 = int((x_center - box_width / 2) * width)
        y1 = int((y_center - box_height / 2) * height)
        x2 = int((x_center + box_width / 2) * width)
        y2 = int((y_center + box_height / 2) * height)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 220, 0), max(2, width // 300))
    return image


for split in ('train', 'val', 'test'):
    image_paths = sorted((YOLO_DATASET_ROOT / 'images' / split).iterdir())
    label_paths = sorted((YOLO_DATASET_ROOT / 'labels' / split).glob('*.txt'))
    if len(image_paths) != len(label_paths):
        raise RuntimeError(f'Image/label count mismatch in {split}')
    print(f'{split}: {len(image_paths)} image-label pairs')

preview_paths = random.Random(SEED).sample(sorted((YOLO_DATASET_ROOT / 'images' / 'val').iterdir()), k=min(6, len(val_records)))
for image_path in preview_paths:
    annotated = draw_yolo_labels(image_path, YOLO_DATASET_ROOT / 'labels' / 'val' / f'{image_path.stem}.txt')
    ok, encoded = cv2.imencode('.jpg', annotated)
    print(image_path.name)
    if ok:
        display(DisplayImage(data=encoded.tobytes(), width=700))

## 6. Train YOLO plate detector
Ubah `RUN_TRAINING = True` pada configuration cell, lalu jalankan cell ini. Mulai dari YOLO11n agar eksperimen cepat. Untuk training cloud/GPU, salin prepared dataset dan notebook tanpa memasukkan output model ke Git.

In [ ]:
training_result = None
if RUN_TRAINING:
    model = YOLO(str(BASE_MODEL_PATH))
    training_result = model.train(
        data=str(DATASET_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
        device=TRAINING_DEVICE, workers=0, seed=SEED, deterministic=True, patience=PATIENCE,
        project=str(EXPERIMENT_ROOT), name=RUN_NAME, exist_ok=True, pretrained=True,
        cache=False, close_mosaic=10, plots=True, verbose=True,
    )
else:
    print('Training skipped. Set RUN_TRAINING=True in the configuration cell when ready.')

## 7. Validate best checkpoint
Validation memakai split `val`; final evaluation memakai split `test` yang tidak digunakan untuk memilih checkpoint.

In [ ]:
BEST_MODEL_PATH = EXPERIMENT_ROOT / RUN_NAME / 'weights' / 'best.pt'
if BEST_MODEL_PATH.is_file():
    trained_model = YOLO(str(BEST_MODEL_PATH))
    validation_metrics = trained_model.val(data=str(DATASET_YAML), split='val', imgsz=IMAGE_SIZE, device=TRAINING_DEVICE, workers=0, plots=True)
    test_metrics = trained_model.val(data=str(DATASET_YAML), split='test', imgsz=IMAGE_SIZE, device=TRAINING_DEVICE, workers=0, plots=True)
    print(f'Validation mAP50: {validation_metrics.box.map50:.4f}')
    print(f'Validation mAP50-95: {validation_metrics.box.map:.4f}')
    print(f'Test mAP50: {test_metrics.box.map50:.4f}')
    print(f'Test mAP50-95: {test_metrics.box.map:.4f}')
else:
    trained_model = None
    print(f'No trained checkpoint yet: {BEST_MODEL_PATH}')

## 8. Inspect predictions on held-out test images

In [ ]:
if trained_model is not None:
    test_images = sorted((YOLO_DATASET_ROOT / 'images' / 'test').iterdir())[:12]
    predictions = trained_model.predict(test_images, conf=0.25, imgsz=IMAGE_SIZE, device=TRAINING_DEVICE, verbose=False)
    for prediction in predictions:
        annotated = prediction.plot()
        ok, encoded = cv2.imencode('.jpg', annotated)
        print(f'{Path(prediction.path).name}: {len(prediction.boxes)} detection(s)')
        if ok:
            display(DisplayImage(data=encoded.tobytes(), width=800))
else:
    print('Train the model first to inspect predictions.')

## 9. Read plate text from YOLO crops

EasyOCR hanya menerima crop hasil bounding box YOLO. Beberapa preprocessing dicoba, fragmen teks dikelompokkan per baris, lalu kandidat dipilih menggunakan pola nomor polisi Indonesia. Detection confidence dan OCR confidence tetap dilaporkan terpisah.

In [ ]:
import re
from time import perf_counter

import easyocr

OCR_ALLOWLIST = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
PLATE_PATTERN = re.compile(r'^[A-Z]{1,3}[0-9]{1,4}[A-Z]{0,3}$')
OCR_USE_GPU = torch.cuda.is_available()
ocr_reader = easyocr.Reader(['en'], gpu=OCR_USE_GPU)


def normalize_plate_text(value: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', value.upper())


def plate_text_score(text: str, confidence: float) -> float:
    score = confidence
    if PLATE_PATTERN.fullmatch(text):
        score += 0.30
    if any(character.isalpha() for character in text) and any(character.isdigit() for character in text):
        score += 0.05
    if not 3 <= len(text) <= 10:
        score -= 0.20
    return score


def group_ocr_lines(detections: list) -> list[list]:
    ordered = sorted(detections, key=lambda item: float(np.mean([point[1] for point in item[0]])))
    lines = []
    for detection in ordered:
        box = detection[0]
        center_y = float(np.mean([point[1] for point in box]))
        height = max(point[1] for point in box) - min(point[1] for point in box)
        target = next((line for line in lines if abs(center_y - line['center_y']) <= 0.60 * max(height, line['height'])), None)
        if target is None:
            lines.append({'center_y': center_y, 'height': height, 'items': [detection]})
        else:
            target['items'].append(detection)
    return [sorted(line['items'], key=lambda item: min(point[0] for point in item[0])) for line in lines]


def best_candidate_from_detections(detections: list) -> tuple[str, float]:
    if not detections:
        return '', 0.0
    lines = group_ocr_lines(detections)
    candidates = []
    for start in range(len(lines)):
        for line_count in (1, 2):
            selected_lines = lines[start:start + line_count]
            if len(selected_lines) != line_count:
                continue
            items = [item for line in selected_lines for item in line]
            text = normalize_plate_text(''.join(item[1] for item in items))
            confidence = float(np.mean([item[2] for item in items]))
            if text:
                candidates.append((text, confidence))
    return max(candidates, key=lambda item: plate_text_score(item[0], item[1]), default=('', 0.0))


def plate_preprocessing_variants(crop: np.ndarray) -> list[tuple[str, np.ndarray]]:
    height = crop.shape[0]
    scale = max(1.0, 112 / max(height, 1))
    resized = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    _, otsu = cv2.threshold(clahe, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return [('original', resized), ('clahe', clahe), ('otsu', otsu)]


def read_plate_crop(crop: np.ndarray) -> dict:
    best = {'text': '', 'confidence': 0.0, 'variant': 'none', 'score': float('-inf')}
    for variant_name, variant in plate_preprocessing_variants(crop):
        detections = ocr_reader.readtext(variant, detail=1, paragraph=False, allowlist=OCR_ALLOWLIST)
        text, confidence = best_candidate_from_detections(detections)
        score = plate_text_score(text, confidence)
        if score > best['score']:
            best = {'text': text, 'confidence': confidence, 'variant': variant_name, 'score': score}
        if PLATE_PATTERN.fullmatch(text) and confidence >= 0.90:
            break
    return best


def crop_box_with_margin(image: np.ndarray, xyxy: list[float], margin: float = 0.08) -> tuple[np.ndarray, tuple[int, int, int, int]]:
    image_height, image_width = image.shape[:2]
    x1, y1, x2, y2 = xyxy
    pad_x, pad_y = (x2 - x1) * margin, (y2 - y1) * margin
    left, top = max(0, int(x1 - pad_x)), max(0, int(y1 - pad_y))
    right, bottom = min(image_width, int(x2 + pad_x)), min(image_height, int(y2 + pad_y))
    return image[top:bottom, left:right].copy(), (left, top, right, bottom)


def detect_and_read_plates(image: np.ndarray, detector: YOLO, detection_confidence: float = 0.40) -> tuple[np.ndarray, list[dict]]:
    prediction = detector.predict(image, conf=detection_confidence, imgsz=IMAGE_SIZE, device=TRAINING_DEVICE, verbose=False)[0]
    annotated = image.copy()
    readings = []
    boxes = prediction.boxes if prediction.boxes is not None else []
    ordered_boxes = sorted(boxes, key=lambda box: float(box.xyxy[0][0]))
    for index, box in enumerate(ordered_boxes, start=1):
        crop, (x1, y1, x2, y2) = crop_box_with_margin(image, box.xyxy[0].tolist())
        if crop.size == 0:
            continue
        started_at = perf_counter()
        ocr_result = read_plate_crop(crop)
        ocr_ms = (perf_counter() - started_at) * 1000
        detection_conf = float(box.conf[0])
        crop_height, crop_width = crop.shape[:2]
        quality_status = 'LOW_RESOLUTION' if crop_width < 100 or crop_height < 30 else 'OK'
        text = ocr_result['text'] or 'UNREADABLE'
        readings.append({
            'index': index, 'text': ocr_result['text'], 'detection_confidence': detection_conf,
            'ocr_confidence': ocr_result['confidence'], 'variant': ocr_result['variant'],
            'ocr_ms': ocr_ms, 'xyxy': (x1, y1, x2, y2), 'crop': crop,
            'crop_width': crop_width, 'crop_height': crop_height, 'quality_status': quality_status,
        })
        color = (0, 180, 0) if ocr_result['text'] else (0, 165, 255)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, max(2, image.shape[1] // 500))
        label = f'{text} | det {detection_conf:.0%} | ocr {ocr_result["confidence"]:.0%}'
        label_y = max(25, y1 - 10)
        cv2.putText(annotated, label, (x1, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2, cv2.LINE_AA)
    return annotated, readings


print('YOLO crop-to-OCR pipeline is ready.')

## 10. End-to-end sample on held-out scenes
Jalankan beberapa gambar scene dari split test untuk memeriksa aliran deteksi dan pembacaan. Hasil OCR belum memiliki ground truth teks untuk semua scene, sehingga bagian ini adalah inspeksi kualitatif.

In [ ]:
END_TO_END_SAMPLE_COUNT = 4
RUN_END_TO_END_SAMPLE = True

if RUN_END_TO_END_SAMPLE and trained_model is not None:
    scene_test_records = [record for record in test_source_records if record['scene_type'] == 'scene'][:END_TO_END_SAMPLE_COUNT]
    for record in scene_test_records:
        image = cv2.imread(str(record['image_path']))
        annotated, readings = detect_and_read_plates(image, trained_model)
        ok, encoded = cv2.imencode('.jpg', annotated)
        print(f'\n{record["image_path"].name} | detected plates: {len(readings)}')
        if ok:
            display(DisplayImage(data=encoded.tobytes(), width=900))
        for reading in readings:
            print(f"Plate {reading['index']}: {reading['text'] or 'UNREADABLE'} | det={reading['detection_confidence']:.2%} | ocr={reading['ocr_confidence']:.2%} | crop={reading['crop_width']}x{reading['crop_height']} | {reading['quality_status']} | {reading['ocr_ms']:.1f} ms")
elif trained_model is None:
    print(f'Trained model not found: {BEST_MODEL_PATH}')
else:
    print('End-to-end sample skipped.')

## 11. Upload a vehicle image for end-to-end testing
Upload gambar kendaraan utuh. Notebook mendeteksi plat dengan YOLO, menampilkan crop setiap deteksi, lalu mencoba membaca nomor polisi dengan OCR.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

detector_upload = widgets.FileUpload(accept='image/*', multiple=False, description='Choose image')
detector_button = widgets.Button(description='Detect and read', button_style='primary', icon='search')
detector_output = widgets.Output()


def first_uploaded_file(value):
    if isinstance(value, dict):
        return next(iter(value.values()), None)
    return value[0] if value else None


def detect_uploaded_image(_):
    with detector_output:
        clear_output(wait=True)
        if trained_model is None:
            print(f'Trained model not found: {BEST_MODEL_PATH}')
            return
        payload = first_uploaded_file(detector_upload.value)
        if payload is None:
            print('Choose an image first.')
            return
        image = cv2.imdecode(np.frombuffer(bytes(payload['content']), dtype=np.uint8), cv2.IMREAD_COLOR)
        if image is None:
            print('The uploaded file is not a readable image.')
            return
        started_at = perf_counter()
        annotated, readings = detect_and_read_plates(image, trained_model)
        total_ms = (perf_counter() - started_at) * 1000
        ok, encoded = cv2.imencode('.jpg', annotated)
        if ok:
            display(DisplayImage(data=encoded.tobytes(), width=900))
        print(f'Detected plates: {len(readings)} | total processing: {total_ms:.1f} ms')
        for reading in readings:
            print(f"\nPlate {reading['index']}: {reading['text'] or 'UNREADABLE'}")
            print(f"Detection confidence: {reading['detection_confidence']:.2%}")
            print(f"OCR confidence: {reading['ocr_confidence']:.2%}")
            print(f"Crop quality: {reading['crop_width']}x{reading['crop_height']} | {reading['quality_status']}")
            print(f"OCR preprocessing: {reading['variant']} | OCR time: {reading['ocr_ms']:.1f} ms")
            crop_ok, crop_encoded = cv2.imencode('.jpg', reading['crop'])
            if crop_ok:
                display(DisplayImage(data=crop_encoded.tobytes(), width=500))


detector_button.on_click(detect_uploaded_image)
display(widgets.VBox([detector_upload, detector_button, detector_output]))

## 12. Export deployment model
ONNX cocok sebagai baseline portable untuk cloud/CPU inference. Export hanya dilakukan saat `RUN_EXPORT = True`.

In [ ]:
if RUN_EXPORT:
    if trained_model is None:
        raise FileNotFoundError(f'Train a checkpoint before export: {BEST_MODEL_PATH}')
    export_path = trained_model.export(format='onnx', imgsz=IMAGE_SIZE, dynamic=True, simplify=True)
    print(f'Exported model: {export_path}')
else:
    print('Export skipped. Set RUN_EXPORT=True when the trained checkpoint is ready.')

## 13. Acceptance criteria and next data iteration

Model belum dianggap siap hanya karena mAP tinggi pada dataset ini. Minimal evaluasi berikut diperlukan:

- mAP50 dan recall pada held-out test.
- Recall terpisah untuk mobil, motor, siang, malam, glare, blur, dan jarak berbeda.
- Ukuran plat dalam piksel pada capture zone aktual.
- False positive pada kendaraan tanpa plat terlihat dan objek mirip plat.
- End-to-end exact-match setelah crop YOLO diteruskan ke recognizer.

Prioritas data berikutnya adalah foto kendaraan utuh dari posisi IP Camera yang direncanakan. Tambahkan hard negatives dan variasi kondisi nyata, lalu train versi baru tanpa mengubah final test set.